In [1]:
import conllu
import pandas as pd

In [2]:
sentences = []
with open("train_nlprepl-ud.conllu", "r", encoding="utf-8") as f:
    for sentence in conllu.parse_incr(f):
        sentences.append(sentence)

In [81]:
from dataclasses import dataclass

@dataclass
class Word:
    lemma: str
    fin_sg_pri_perf: str
    fin_sg_pri_imperf: str
    fin_sg_sec_perf: str
    fin_sg_sec_imperf: str
    fin_sg_ter_perf: str
    praet_sg_f_perf: str
    praet_sg_m1m2m3_perf: str
    fin_pl_pri_perf: str


    def by_tag(self, tag):
        if tag == "fin:sg:pri:perf":
            return self.fin_sg_pri_perf
        elif tag == "fin:sg:pri:imperf":
            return self.fin_sg_pri_perf
        elif tag == "fin:sg:sec:perf":
            return self.fin_sg_sec_perf
        elif tag == "fin:sg:sec:imperf":
            return self.fin_sg_sec_imperf
        elif tag == "fin:pl:pri:perf":
            return self.fin_pl_pri_perf
        else:
            raise Exception("Unknown tag {}".format(tag))


In [82]:
dict_df = pd.read_csv("dictionary.v2.csv")
word_by_lemma = {}
for row in dict_df.itertuples(index=False, name=None):
    word = Word(*row)
    word_by_lemma[word.lemma] = word

Predicate - token that:
* is root, or
* is the HEAD of a subject token (can be checked with 'deprel' dependency relation label)

In [18]:
from copy import deepcopy
from tqdm import tqdm

def sentence_text(sent):
    """Conlu library doesn't have 'generate raw text' function"""
    out = []
    for tok in sent:
        if not isinstance(tok["id"], int):
            continue

        out.append(tok["form"])

        misc = tok.get("misc")
        if not misc or misc.get("SpaceAfter") != "No":
            out.append(" ")

    return "".join(out).rstrip()

In [80]:
sentences[7].to_tree().token['xpos']

'fin:sg:ter:perf'

In [88]:
def simple_tag_to_tag_replacement_with_root_predicate(sentences, source_xpos, target_xpos, limit: int =None):
    pairs = []

    for sentence in tqdm(sentences):
        if limit is not None:
            if len(pairs) >= limit:
                break

        sentence = deepcopy(sentence)  # to not break original objects
        root = sentence.to_tree()

        upos = root.token['upos'] # Universal Part-of-Speech
        if upos != 'VERB':
            # For now lets take only root predicates
            continue

        has_subj = False
        for ch in root.children:
            if ch.token['deprel'] == 'nsubj':
                has_subj = True

        if not has_subj:
            continue

        # For now lets take only root predicates
        lemma = root.token["lemma"]
        if lemma not in word_by_lemma:
            continue
        word = word_by_lemma[lemma]

        xpos = root.token['xpos'] # language-specific part-of-speech tag
        if xpos != source_xpos:
            continue

        is_capitalized = root.token['form'][0].isupper()

        target_form = word.by_tag(target_xpos)

        # TODO figure out floats?
        if not isinstance(target_form, str):
            continue

        assert len(target_form) > 0

        if is_capitalized:
            target_form = target_form[:1].upper() + target_form[1:]

        root.token['form'] = target_form
        root.token['xpos'] = target_xpos

        correct_text = sentence.metadata['text']
        incorrect_text = sentence_text(sentence)
        pairs.append((correct_text, incorrect_text))

    return pd.DataFrame(pairs, columns=["correct", "incorrect"])

In [89]:
# TODO inne warianty jak notfin / imperf
# TODO merge into one loop
# TODO trzecia osoba
# jakis elegancki kod od tagów, w generacji dicta obsluzyc kombinatoryke

# TODO wrzucic indeks z pliku zrodlowego (do debugowania potem)

# dfs = []
#
# dfs.append(simple_tag_to_tag_replacement_with_root_predicate(sentences, 'fin:sg:pri:perf', 'fin:sg:sec:perf'))
# dfs.append(simple_tag_to_tag_replacement_with_root_predicate(sentences, 'fin:sg:pri:imperf', 'fin:sg:sec:imperf'))
# dfs.append(simple_tag_to_tag_replacement_with_root_predicate(sentences, 'fin:sg:sec:perf', 'fin:sg:pri:perf'))
dfs.append(simple_tag_to_tag_replacement_with_root_predicate(sentences, 'fin:sg:sec:imperf', 'fin:sg:pri:imperf'))

combined = pd.concat(dfs)
combined.sort_values('correct')

combined.to_csv("a_pri_sec.csv", index=False)

100%|██████████| 69360/69360 [01:18<00:00, 884.77it/s] 


In [85]:

combined = pd.concat(dfs)
combined.sort_values('correct')

combined.to_csv("a_pri_sec.csv", index=False)

In [73]:
# TODO inne warianty jak notfin / imperf

sg_to_pl = simple_tag_to_tag_replacement_with_root_predicate(sentences, 'fin:sg:pri:perf', 'fin:pl:pri:perf')
pls_to_sg = simple_tag_to_tag_replacement_with_root_predicate(sentences, 'fin:pl:pri:perf', 'fin:sg:pri:perf')

combined = pd.concat([sg_to_pl, pls_to_sg])
combined.sort_values('correct')

combined.to_csv("b_sg_pl.csv", index=False)

100%|██████████| 69360/69360 [01:23<00:00, 826.06it/s] 
